# 14 · Versioned artifacts

Artifacts store files separately from session state. A tool saves an invoice CSV and another cell loads the resulting version from ADK's artifact service.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** Lite model calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
import csv, io
from google.adk.tools import ToolContext

async def export_invoice(order_id: str, tool_context: ToolContext) -> dict:
    """Create a demo invoice CSV artifact from an existing order."""
    order = get_order(order_id)
    if "error" in order:
        return order
    buffer = io.StringIO()
    writer = csv.writer(buffer)
    writer.writerow(["product_id", "quantity", "unit_price"])
    for item in order["items"]:
        writer.writerow([item["product_id"], item["quantity"], item["unit_price"]])
    filename = order["invoice_id"] + ".csv"
    version = await tool_context.save_artifact(filename, types.Part.from_bytes(
        data=buffer.getvalue().encode(), mime_type="text/csv"))
    return {"filename": filename, "version": version, "total": order["total"]}

lab = await make_lab(llm("invoice_exporter", "Call export_invoice once, then report the filename.", tools=[export_invoice]))
await ask(lab, "Export the invoice for O1001.")
artifact = await lab.runner.artifact_service.load_artifact(
    app_name=lab.runner.app_name, user_id=lab.user_id, session_id=lab.session.id, filename="INV1001.csv")
print(artifact.inline_data.data.decode())


## Try it

Export the same filename twice and inspect its version numbers. In-memory artifacts vanish with the kernel; cloud artifact storage is optional.


## Reference

[Official ADK documentation](https://adk.dev/artifacts/). Shared configuration: `config.json`. No environment activation or login cells are needed.
